# Notebook 5: Preprocessing
## Car Price Prediction Project

**Objective:** Prepare data for modeling — encode categoricals, scale numericals, split into train/test sets.

---

## 5.1 Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/content/drive/MyDrive/car_price_engineered.csv')
print(f"Loaded: {df.shape}")
df.head()

## 5.2 Identify Column Types

In [ ]:
# Separate column types
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Remove target and log_price from feature lists
target = 'price'
cols_to_exclude = ['price', 'log_price']
num_features = [c for c in num_cols if c not in cols_to_exclude]

print(f"Categorical features ({len(cat_cols)}): {cat_cols}")
print(f"\nNumerical features ({len(num_features)}): {num_features}")
print(f"\nTarget: {target}")

## 5.3 Drop Helper / Intermediate Columns

We drop columns that were only used to create other features (e.g., `brand` was used to create `brand_tier` and `brand_avg_price`).

In [ ]:
# Drop intermediate columns that shouldn't be model features
# Keep log_price separate — we'll decide later if we use it as target
drop_cols = ['brand', 'log_price']  # brand is replaced by brand_tier and brand_avg_price

# Also drop binned columns (they were for EDA; originals + bins together cause redundancy)
bin_cols = ['hp_bin', 'enginesize_bin', 'weight_bin']
drop_cols.extend(bin_cols)

# Only drop columns that exist
drop_cols = [c for c in drop_cols if c in df.columns]
print(f"Dropping: {drop_cols}")

df.drop(columns=drop_cols, inplace=True)
print(f"Shape after drop: {df.shape}")

## 5.4 One-Hot Encode Categorical Features

In [ ]:
# Identify remaining categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns to encode: {cat_cols}")
for col in cat_cols:
    print(f"  {col}: {df[col].nunique()} unique → {df[col].unique().tolist()}")

In [ ]:
# One-hot encode with drop_first=True to avoid dummy variable trap
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
print(f"Shape after encoding: {df_encoded.shape}")
print(f"\nNew columns: {df_encoded.columns.tolist()}")

## 5.5 Define Features (X) and Target (y)

In [ ]:
X = df_encoded.drop(columns=['price'])
y = df_encoded['price']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nFeature names ({X.shape[1]}):")
for i, col in enumerate(X.columns, 1):
    print(f"  {i}. {col}")

## 5.6 Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nTrain price stats:")
print(y_train.describe())
print(f"\nTest price stats:")
print(y_test.describe())

## 5.7 Feature Scaling

In [ ]:
# StandardScaler — fit on train, transform both
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Scaling applied (StandardScaler).")
print(f"\nTrain scaled sample:")
X_train_scaled.head()

## 5.8 Verify No Data Leakage

In [ ]:
# Ensure train and test indices don't overlap
overlap = set(X_train.index) & set(X_test.index)
print(f"Index overlap between train and test: {len(overlap)} (should be 0)")

# Ensure target is not in features
assert 'price' not in X_train.columns, "LEAK: price in features!"
print("No data leakage detected.")

## 5.9 Save Preprocessed Data

In [ ]:
# Save all splits to Google Drive for the modeling notebook
X_train_scaled.to_csv('/content/drive/MyDrive/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('/content/drive/MyDrive/X_test_scaled.csv', index=False)
y_train.to_csv('/content/drive/MyDrive/y_train.csv', index=False)
y_test.to_csv('/content/drive/MyDrive/y_test.csv', index=False)

# Also save unscaled versions (for tree-based models that don't need scaling)
X_train.to_csv('/content/drive/MyDrive/X_train.csv', index=False)
X_test.to_csv('/content/drive/MyDrive/X_test.csv', index=False)

# Save the full encoded dataframe for PyCaret
df_encoded.to_csv('/content/drive/MyDrive/car_price_encoded.csv', index=False)

print("All preprocessed files saved:")
print("  - X_train_scaled.csv / X_test_scaled.csv (for Linear Regression)")
print("  - X_train.csv / X_test.csv (for tree-based models)")
print("  - y_train.csv / y_test.csv")
print("  - car_price_encoded.csv (full dataset for PyCaret)")

---
## Preprocessing Summary

| Step | Action |
|------|--------|
| 1 | Dropped intermediate/helper columns (`brand`, `log_price`, bins) |
| 2 | One-hot encoded categorical features (with `drop_first=True`) |
| 3 | Split into 80% train / 20% test (`random_state=42`) |
| 4 | Applied StandardScaler (fit on train, transform both) |
| 5 | Verified no data leakage |

**Next step →** Notebook 06: Train Models